# Per-chore models: dishes and trash separately

The chore type (dishes vs trash) is never wrong, so every error is *Complete vs Incomplete within one chore*.
Here each chore is its own binary problem with its own feature / model search:

1. **Chore router** – logistic regression on DINOv2 (checked with CV; it is perfect).
2. **Dishes**: Complete vs Incomplete on the 34 competition dishes photos (+ external #912 in training only).
3. **Trash**: Complete vs Incomplete on the 32 competition trash photos (+ 11 external trash images in training only), also trying the CDCM overflow / tile scores.
4. Combine the best dishes model and the best trash model and compare with the current best 4-class model (DINOv2 + tiles RBF SVM, CV 0.890).

**CV protocol** (same as `aux_cv.py`): 10 repeats × stratified 5-fold, grouped by photo (each row is one photo; augmented views
only ever join the training fold of their own photo). External images are only in training folds; scores use competition photos only.
Metric per chore = macro F1 over its two classes (that is exactly that chore's share of the overall 4-class macro F1).

In [1]:
import time, warnings
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn
from scipy.special import expit
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from extract_features import CLASSES, unit

warnings.filterwarnings("ignore")
torch.set_num_threads(4)
pd.set_option("display.width", 220); pd.set_option("display.max_rows", 200)
ROOT = Path(".")
REPEATS, ALPHA = 10, 1.0
BACKBONES = ["dinov2_l", "eva02_l", "siglip_so400m", "vith14_laion", "vitl14_openai"]
AUGMENTED = ["dinov2_l", "eva02_l", "siglip_so400m"]          # backbones with 20 augmented views per photo

train = pd.read_csv(ROOT / "data/train.csv")    # 66 competition photos + 12 external
test = pd.read_csv(ROOT / "data/test.csv")
ALL = pd.concat([train.id, test.id]).values
y4 = train.label.map(CLASSES.index).values
is_ext = train.image_file.str.contains("_external").values
chore = np.where(train.label.str.startswith("Washing"), "dishes", "trash")
print(pd.crosstab(train.label, np.where(is_ext, "external", "competition")))

col_0                        competition  external
label                                             
Taking Out Trash|Complete             13         7
Taking Out Trash|Incomplete           20         4
Washing Dishes|Complete               17         1
Washing Dishes|Incomplete             16         0


In [2]:
def rows(z, ids):
    pos = {i: k for k, i in enumerate(z["ids"])}
    return [pos[i] for i in ids]

def full(b, ids):                      # image + mirror, L2-normalised (the 'full' view used everywhere)
    z = np.load(ROOT / f"features/{b}.npz"); r = rows(z, ids)
    return unit(z["img"][r] + z["img_flip"][r])

def aug(b, ids):                       # (N, 20, D) random augmented views
    z = np.load(ROOT / f"features/{b}_aug.npz")
    return unit(z["aug"][rows(z, ids)])

def scores(path, keys, ids):
    z = np.load(ROOT / path); r = rows(z, ids)
    return np.stack([z[k][r] for k in keys], 1)

FULL = {b: full(b, ALL) for b in BACKBONES}
AUG = {b: aug(b, ALL) for b in AUGMENTED}
TILES = scores("features/overflow_tiles.npz",
               ["dinov2_l_tile_max", "dinov2_l_tile_top2", "siglip_so400m_tile_max", "siglip_so400m_tile_top2"], ALL)
OVERFLOW = scores("features/overflow_score.npz",
                  ["dinov2_l_full", "dinov2_l_max7", "siglip_so400m_full", "siglip_so400m_max7"], ALL)
z = np.load(ROOT / "features/rand_aug_dinov2_l.npz"); RAND = unit(z["img"] + z["mirror"])[rows(z, train.id)]  # train only
NTR = len(train)

SHORT = {"dinov2_l": "dinov2", "eva02_l": "eva02", "siglip_so400m": "siglip", "vith14_laion": "vith14", "vitl14_openai": "vitl14"}
FEATS = {SHORT[b]: [b] for b in BACKBONES}
FEATS["dinov2+siglip"] = ["dinov2_l", "siglip_so400m"]
FEATS["5-backbone concat"] = BACKBONES

def build(feat, extra=None):
    """Feature matrix for train+test rows (full view), optional overflow/tile scores appended."""
    X = unit(np.hstack([FULL[b] for b in FEATS[feat]]))
    if extra == "tiles": X = np.hstack([X, ALPHA * TILES])
    if extra == "overflow": X = np.hstack([X, ALPHA * OVERFLOW])
    if extra == "tiles+overflow": X = np.hstack([X, ALPHA * TILES, ALPHA * OVERFLOW])
    return X

def views(feat, extra, mode):
    """Extra training views (N_train+test, V, D) for 'aug' (20 aug views) or 'rand10' (10 random crops, DINOv2 only)."""
    if mode == "aug":
        V = unit(np.concatenate([AUG[b] for b in FEATS[feat]], -1))
    elif mode == "rand10":
        V = np.full((len(ALL), 10, RAND.shape[-1]), np.nan); V[:NTR] = RAND[:, :10]
    else:
        return None
    S = {"tiles": TILES, "overflow": OVERFLOW, "tiles+overflow": np.hstack([TILES, OVERFLOW])}.get(extra)
    if S is not None:           # the photo-level scores are copied onto every view of that photo
        V = np.concatenate([V, np.repeat(ALPHA * S[:, None], V.shape[1], 1)], -1)
    return V
print({k: v.shape for k, v in FULL.items()}, TILES.shape, RAND.shape)

{'dinov2_l': (114, 2048), 'eva02_l': (114, 768), 'siglip_so400m': (114, 1152), 'vith14_laion': (114, 1024), 'vitl14_openai': (114, 768)} (114, 4) (78, 60, 2048)


## Models
Binary classifiers (positive class = Incomplete), all with balanced class weights. The MLP is the same small network as in the
comparison notebooks, with 2 outputs. RBF/linear SVM probabilities = sigmoid of the decision value (only used for averaging).

In [3]:
class TorchMLP:
    def __init__(self, steps=400, batch=32, seed=0): self.steps, self.batch, self.seed = steps, batch, seed
    def fit(self, X, yy):
        torch.manual_seed(self.seed)
        self.mu, self.sd = X.mean(0), X.std(0) + 1e-6
        self.net = nn.Sequential(nn.Dropout(0.2), nn.Linear(X.shape[1], 256), nn.GELU(), nn.Dropout(0.5), nn.Linear(256, 2))
        w = torch.tensor(len(yy) / (2 * np.bincount(yy, minlength=2).clip(1)), dtype=torch.float32)
        loss_fn = nn.CrossEntropyLoss(weight=w, label_smoothing=0.1)
        opt = torch.optim.AdamW(self.net.parameters(), lr=1e-3, weight_decay=1e-2)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, self.steps)
        Xt, yt = torch.tensor((X - self.mu) / self.sd, dtype=torch.float32), torch.tensor(yy)
        g = torch.Generator().manual_seed(self.seed); self.net.train()
        for _ in range(self.steps):
            b = torch.randint(len(Xt), (min(self.batch, len(Xt)),), generator=g)
            opt.zero_grad(); loss_fn(self.net(Xt[b]), yt[b]).backward(); opt.step(); sched.step()
        return self
    def predict_proba(self, X):
        self.net.eval()
        with torch.no_grad():
            return torch.softmax(self.net(torch.tensor((X - self.mu) / self.sd, dtype=torch.float32)), 1).numpy()

BAL = dict(class_weight="balanced")
MODELS = {
    "logreg C=0.3": lambda: LogisticRegression(C=0.3, max_iter=5000, **BAL),
    "logreg C=1":   lambda: LogisticRegression(C=1, max_iter=5000, **BAL),
    "logreg C=10":  lambda: LogisticRegression(C=10, max_iter=5000, **BAL),
    "linear SVM":   lambda: SVC(kernel="linear", C=1, **BAL),
    "RBF SVM C=1":  lambda: SVC(kernel="rbf", C=1, gamma="scale", **BAL),
    "RBF SVM C=3":  lambda: SVC(kernel="rbf", C=3, gamma="scale", **BAL),
    "RBF SVM C=10": lambda: SVC(kernel="rbf", C=10, gamma="scale", **BAL),
    "kNN k=5":      lambda: KNeighborsClassifier(5, metric="cosine", weights="distance"),
    "extra trees":  lambda: ExtraTreesClassifier(300, max_features="sqrt", n_jobs=-1, random_state=0, **BAL),
    "MLP":          lambda: TorchMLP(),
}

def p_incomplete(clf, X):
    return clf.predict_proba(X)[:, 1] if hasattr(clf, "predict_proba") else expit(clf.decision_function(X))

## CV helper
`run_chore` runs every (features, extra scores, view mode, model) combination on one chore and keeps the out-of-fold
p(Incomplete) for each repeat, so the best dishes and best trash models can be combined afterwards.

In [4]:
def chore_splits(name, seed=0):
    comp = np.where((chore == name) & ~is_ext)[0]
    ext = np.where((chore == name) & is_ext)[0]
    yb = (y4 % 2)                      # 0 = Complete, 1 = Incomplete
    sp = [(np.r_[comp[tr], ext], comp[va]) for tr, va in
          RepeatedStratifiedKFold(n_splits=5, n_repeats=REPEATS, random_state=seed).split(comp, yb[comp])]
    return comp, sp, yb

def fit_predict(make, X, V, tr, va, yb):
    Xtr, ytr = X[tr], yb[tr]
    if V is not None:
        Vt = V[tr]; keep = ~np.isnan(Vt).any(-1)
        Xtr = np.vstack([Xtr, Vt[keep]]); ytr = np.concatenate([ytr, np.repeat(ytr, Vt.shape[1])[keep.ravel()]])
    return p_incomplete(make().fit(Xtr, ytr), X[va])

def run_chore(name, configs, models=MODELS):
    comp, sp, yb = chore_splits(name)
    res, oof = [], {}
    for feat, extra, mode in configs:
        X, V = build(feat, extra)[:NTR], views(feat, extra, mode)
        V = None if V is None else V[:NTR]
        for mname, make in models.items():
            if mode != "full" and mname in ("MLP", "kNN k=5"): continue    # keep runtime sane
            t0 = time.time(); P = np.full((REPEATS, NTR), np.nan)
            for k, (tr, va) in enumerate(sp):
                P[k // 5, va] = fit_predict(make, X, V, tr, va, yb)
            f1 = [f1_score(yb[comp], (p[comp] > 0.5).astype(int), average="macro") for p in P]
            wrong = np.mean([((p[comp] > 0.5) != yb[comp]).sum() for p in P])
            key = (feat, extra or "-", mode, mname)
            oof[key] = P
            res.append(dict(features=feat, extra=extra or "-", views=mode, model=mname,
                            F1=np.mean(f1), std=np.std(f1), wrong=wrong, secs=time.time() - t0))
        print(f"{name}: {feat:18s} {extra or '-':15s} {mode:7s} done; best so far "
              f"{max(r['F1'] for r in res):.3f}", flush=True)
    return pd.DataFrame(res).sort_values("F1", ascending=False).reset_index(drop=True), oof, comp, yb

## 0. Chore router
Logistic regression on DINOv2 full view, dishes vs trash. If this is perfect in CV, combining the two per-chore models loses nothing.

In [5]:
Xc = build("dinov2")
yc = (chore == "trash").astype(int)
comp_all = np.where(~is_ext)[0]
errs = []
for tr, va in RepeatedStratifiedKFold(n_splits=5, n_repeats=REPEATS, random_state=0).split(comp_all, y4[comp_all]):
    tr = np.r_[comp_all[tr], np.where(is_ext)[0]]; va = comp_all[va]
    errs.append((LogisticRegression(C=1, max_iter=5000).fit(Xc[tr], yc[tr]).predict(Xc[va]) != yc[va]).sum())
router = LogisticRegression(C=1, max_iter=5000).fit(Xc[:NTR], yc)
test_chore = np.where(router.predict(Xc[NTR:]) == 1, "trash", "dishes")
p_te = router.predict_proba(Xc[NTR:])[:, 1]
print(f"chore errors in CV: {sum(errs)} over {REPEATS} x 66 predictions")
print("test chore counts:", pd.Series(test_chore).value_counts().to_dict(),
      f"| least confident test photo: p(trash) closest to 0.5 = {p_te[np.argmin(abs(p_te - 0.5))]:.3f}")

chore errors in CV: 10 over 10 x 66 predictions
test chore counts: {'dishes': 19, 'trash': 17} | least confident test photo: p(trash) closest to 0.5 = 0.379


## 1. Dishes: Complete vs Incomplete

In [6]:
feats = list(FEATS)
dish_cfg = [(f, None, "full") for f in feats] + [(SHORT[b], None, "aug") for b in AUGMENTED] + [("dinov2", None, "rand10")]
t0 = time.time()
dish_res, dish_oof, dish_comp, yb = run_chore("dishes", dish_cfg)
print(f"{len(dish_res)} combos in {time.time() - t0:.0f}s")
dish_res.drop(columns="secs").head(20).round(3)

dishes: dinov2             -               full    done; best so far 0.845


dishes: eva02              -               full    done; best so far 0.845


dishes: siglip             -               full    done; best so far 0.845


dishes: vith14             -               full    done; best so far 0.845


dishes: vitl14             -               full    done; best so far 0.845


dishes: dinov2+siglip      -               full    done; best so far 0.866


... (3 outputs left out)

dishes: siglip             -               aug     done; best so far 0.878


dishes: dinov2             -               rand10  done; best so far 0.878


102 combos in 425s


,features,extra,views,model,F1,std,wrong
0,dinov2,-,aug,logreg C=0.3,0.878,0.039,4.0
1,dinov2,-,rand10,logreg C=10,0.869,0.039,4.3
2,dinov2,-,aug,logreg C=1,0.869,0.043,4.3
3,dinov2,-,rand10,linear SVM,0.866,0.037,4.4
4,dinov2+siglip,-,full,logreg C=0.3,0.866,0.037,4.4
5,dinov2+siglip,-,full,extra trees,0.865,0.041,4.4
6,dinov2,-,aug,logreg C=10,0.863,0.042,4.5
7,dinov2+siglip,-,full,logreg C=1,0.863,0.037,4.5
8,dinov2+siglip,-,full,logreg C=10,0.860,0.031,4.6
9,dinov2,-,aug,linear SVM,0.860,0.047,4.6


In [7]:
print("Dishes – best model per feature set / view mode")
dish_res.loc[dish_res.groupby(["features", "views"]).F1.idxmax()].sort_values("F1", ascending=False).drop(columns="secs").round(3)

Dishes – best model per feature set / view mode


,features,extra,views,model,F1,std,wrong
0,dinov2,-,aug,logreg C=0.3,0.878,0.039,4.0
1,dinov2,-,rand10,logreg C=10,0.869,0.039,4.3
4,dinov2+siglip,-,full,logreg C=0.3,0.866,0.037,4.4
21,dinov2,-,full,logreg C=10,0.845,0.040,5.1
23,siglip,-,full,MLP,0.842,0.030,5.2
28,5-backbone concat,-,full,logreg C=10,0.835,0.028,5.4
41,siglip,-,aug,linear SVM,0.815,0.048,6.0
44,eva02,-,aug,RBF SVM C=1,0.814,0.051,6.1
52,eva02,-,full,MLP,0.808,0.043,6.3
57,vith14,-,full,MLP,0.805,0.039,6.4


In [8]:
print("Dishes – mean F1 per model (over all feature sets)")
dish_res.groupby("model").F1.agg(["mean", "max"]).sort_values("mean", ascending=False).round(3)

Dishes – mean F1 per model (over all feature sets)


,mean,max
model,,
logreg C=10,0.812,0.869
MLP,0.806,0.851
logreg C=1,0.804,0.869
RBF SVM C=3,0.802,0.860
logreg C=0.3,0.800,0.878
RBF SVM C=10,0.799,0.860
extra trees,0.793,0.865
RBF SVM C=1,0.785,0.857
linear SVM,0.778,0.866


## 2. Trash: Complete vs Incomplete
Same grid, plus the CDCM 'dirty bin' scores (whole-image overflow score, tile scores, both) appended to the features.

In [9]:
trash_cfg = []
for extra in (None, "tiles", "overflow", "tiles+overflow"):
    trash_cfg += [(f, extra, "full") for f in feats] + [(SHORT[b], extra, "aug") for b in AUGMENTED] + [("dinov2", extra, "rand10")]
t0 = time.time()
trash_res, trash_oof, trash_comp, _ = run_chore("trash", trash_cfg)
print(f"{len(trash_res)} combos in {time.time() - t0:.0f}s")
trash_res.drop(columns="secs").head(20).round(3)

trash: dinov2             -               full    done; best so far 0.849


trash: eva02              -               full    done; best so far 0.871


trash: siglip             -               full    done; best so far 0.871


trash: vith14             -               full    done; best so far 0.881


trash: vitl14             -               full    done; best so far 0.897


trash: dinov2+siglip      -               full    done; best so far 0.897


... (36 outputs left out)

trash: siglip             tiles+overflow  aug     done; best so far 0.910


trash: dinov2             tiles+overflow  rand10  done; best so far 0.910


408 combos in 2135s


,features,extra,views,model,F1,std,wrong
0,vith14,tiles,full,RBF SVM C=10,0.910,0.035,2.8
1,eva02,overflow,aug,logreg C=1,0.908,0.022,2.9
2,5-backbone concat,tiles,full,RBF SVM C=1,0.907,0.021,2.8
3,eva02,overflow,full,logreg C=10,0.904,0.020,3.0
4,5-backbone concat,tiles,full,RBF SVM C=10,0.901,0.022,3.1
5,eva02,overflow,full,RBF SVM C=3,0.900,0.010,3.1
6,vith14,tiles,full,linear SVM,0.897,0.012,3.2
7,vitl14,-,full,RBF SVM C=1,0.897,0.014,3.3
8,5-backbone concat,overflow,full,logreg C=10,0.895,0.029,3.3
9,vith14,tiles,full,RBF SVM C=3,0.894,0.015,3.3


In [10]:
print("Trash – best model per feature set / extra scores / view mode (top 25)")
trash_res.loc[trash_res.groupby(["features", "extra", "views"]).F1.idxmax()].sort_values("F1", ascending=False).drop(columns="secs").head(25).round(3)

Trash – best model per feature set / extra scores / view mode (top 25)


,features,extra,views,model,F1,std,wrong
0,vith14,tiles,full,RBF SVM C=10,0.910,0.035,2.8
1,eva02,overflow,aug,logreg C=1,0.908,0.022,2.9
2,5-backbone concat,tiles,full,RBF SVM C=1,0.907,0.021,2.8
3,eva02,overflow,full,logreg C=10,0.904,0.020,3.0
7,vitl14,-,full,RBF SVM C=1,0.897,0.014,3.3
8,5-backbone concat,overflow,full,logreg C=10,0.895,0.029,3.3
10,5-backbone concat,tiles+overflow,full,RBF SVM C=3,0.894,0.015,3.3
11,dinov2,tiles,full,RBF SVM C=3,0.892,0.025,3.4
22,dinov2+siglip,tiles,full,RBF SVM C=3,0.888,0.022,3.5
24,dinov2,overflow,full,RBF SVM C=1,0.888,0.015,3.5


In [11]:
print("Trash – does adding the dirty-bin scores help? (mean / max F1 over all features & models)")
trash_res.groupby("extra").F1.agg(["mean", "max"]).round(3)

Trash – does adding the dirty-bin scores help? (mean / max F1 over all features & models)


,mean,max
extra,,
-,0.834,0.897
overflow,0.844,0.908
tiles,0.842,0.910
tiles+overflow,0.828,0.894


## 3. Per-photo view of the best models
Times wrong out of 10 repeats for every competition photo that the best model, or the current 4-class baseline, ever gets wrong.

In [12]:
# current best 4-class model (submitted file submission_dinov2_tiles_svm.csv): DINOv2 full + tiles, RBF SVM C=3, 4 classes,
# same 10x5 4-class splits as aux_cv.py
Xb = build("dinov2", "tiles")[:NTR]
comp_all, ext_all = np.where(~is_ext)[0], np.where(is_ext)[0]
base = np.zeros((REPEATS, NTR), int)
for k, (tr, va) in enumerate(RepeatedStratifiedKFold(n_splits=5, n_repeats=REPEATS, random_state=0).split(comp_all, y4[comp_all])):
    tr = np.r_[comp_all[tr], ext_all]; va = comp_all[va]
    base[k // 5, va] = SVC(kernel="rbf", C=3, gamma="scale", **BAL).fit(Xb[tr], y4[tr]).predict(Xb[va])
base_f1 = [f1_score(y4[comp_all], b[comp_all], average="macro") for b in base]
print(f"4-class baseline CV macro F1 {np.mean(base_f1):.3f} ± {np.std(base_f1):.3f}")
for name, comp in (("dishes", dish_comp), ("trash", trash_comp)):
    yy = y4[comp] % 2
    print(f"  baseline on {name}: F1 {np.mean([f1_score(yy, b[comp] % 2, average='macro') for b in base]):.3f}, "
          f"wrong {np.mean([(b[comp] != y4[comp]).sum() for b in base]):.1f}/{len(comp)}")

def per_photo(res, oof, comp, top=3):
    out = {"label": train.label.values[comp], "baseline": (base[:, comp] != y4[comp]).sum(0)}
    for i in range(top):
        key = tuple(res.loc[i, ["features", "extra", "views", "model"]])
        out[f"#{i+1} " + " | ".join(k for k in key if k != "-")] = ((oof[key][:, comp] > 0.5) != (y4[comp] % 2)).sum(0)
    df = pd.DataFrame(out, index=[f"#{i}" for i in train.id.values[comp]])
    return df[(df.drop(columns="label") > 0).any(axis=1)]

print("\nDishes photos (times wrong of 10):"); display(per_photo(dish_res, dish_oof, dish_comp))
print("Trash photos (times wrong of 10):"); display(per_photo(trash_res, trash_oof, trash_comp))

4-class baseline CV macro F1 0.890 ± 0.026
  baseline on dishes: F1 0.875, wrong 4.1/33
  baseline on trash: F1 0.905, wrong 3.0/33

Dishes photos (times wrong of 10):


,label,baseline,#1 dinov2 | aug | logreg C=0.3,#2 dinov2 | rand10 | logreg C=10,#3 dinov2 | aug | logreg C=1
#2,Washing Dishes|Incomplete,7,2,1,1
#12,Washing Dishes|Complete,1,0,0,0
#31,Washing Dishes|Incomplete,10,10,5,8
#46,Washing Dishes|Incomplete,1,2,1,1
#47,Washing Dishes|Incomplete,1,2,3,2
#48,Washing Dishes|Incomplete,1,3,3,3
#60,Washing Dishes|Incomplete,10,10,10,10
#81,Washing Dishes|Complete,0,0,1,1
#99,Washing Dishes|Complete,2,4,9,9
#101,Washing Dishes|Complete,8,7,10,8


Trash photos (times wrong of 10):


,label,baseline,#1 vith14 | tiles | full | RBF SVM C=10,#2 eva02 | overflow | aug | logreg C=1,#3 5-backbone concat | tiles | full | RBF SVM C=1
#13,Taking Out Trash|Complete,2,0,0,0
#19,Taking Out Trash|Incomplete,5,0,10,0
#26,Taking Out Trash|Complete,10,10,10,10
#61,Taking Out Trash|Complete,0,2,0,6
#63,Taking Out Trash|Complete,1,3,0,1
#65,Taking Out Trash|Incomplete,0,1,0,0
#72,Taking Out Trash|Incomplete,10,9,2,0
#77,Taking Out Trash|Complete,2,3,5,10
#80,Taking Out Trash|Incomplete,0,0,2,0
#89,Taking Out Trash|Complete,0,0,0,1


## 4. Combine: dishes model + trash model → 4-class macro F1
Every competition photo gets one out-of-fold prediction per repeat from its chore's model (the router is perfect in CV),
so the overall 4-class macro F1 per repeat can be computed directly and compared with the baseline.

**Caveat:** picking the top of a big grid is optimistic (winner's curse; with ~34 photos per chore one photo flip ≈ 0.03 F1).
So I also show the combination of models chosen *by rule* (the same 'boring' model family on both chores) and the average of
the top 5 per chore, which is less sensitive to lucky picks.

In [13]:
def combine(dkeys, tkeys):
    """Average p(Incomplete) over the given dishes / trash model keys, then 4-class macro F1 per repeat."""
    pred = np.zeros((REPEATS, NTR), int)
    pd_ = np.mean([dish_oof[k] for k in dkeys], 0); pt = np.mean([trash_oof[k] for k in tkeys], 0)
    pred[:, dish_comp] = (pd_[:, dish_comp] > 0.5).astype(int)          # 0/1 = dishes Complete/Incomplete
    pred[:, trash_comp] = 2 + (pt[:, trash_comp] > 0.5).astype(int)     # 2/3 = trash Complete/Incomplete
    f1 = [f1_score(y4[comp_all], p[comp_all], average="macro") for p in pred]
    return np.mean(f1), np.std(f1)

key = lambda res, i: tuple(res.loc[i, ["features", "extra", "views", "model"]])
rows_ = [{"combination": "4-class baseline (dinov2+tiles RBF SVM)", "CV macro F1": np.mean(base_f1), "std": np.std(base_f1)}]
def add(name, d, t):
    m, s = combine(d, t); rows_.append({"combination": name, "CV macro F1": m, "std": s})
add("best dishes + best trash", [key(dish_res, 0)], [key(trash_res, 0)])
add("top-5 dishes avg + top-5 trash avg", [key(dish_res, i) for i in range(5)], [key(trash_res, i) for i in range(5)])
add("per chore: dinov2 RBF SVM C=3 (dishes) / dinov2+tiles RBF SVM C=3 (trash)",
    [("dinov2", "-", "full", "RBF SVM C=3")], [("dinov2", "tiles", "full", "RBF SVM C=3")])
add("per chore: dinov2 logreg C=1 +rand10 (dishes) / dinov2+tiles RBF SVM C=3 (trash)",
    [("dinov2", "-", "rand10", "logreg C=1")], [("dinov2", "tiles", "full", "RBF SVM C=3")])
add("best dishes + per-chore dinov2+tiles RBF SVM C=3 (trash)", [key(dish_res, 0)], [("dinov2", "tiles", "full", "RBF SVM C=3")])
add("per-chore dinov2 RBF SVM C=3 (dishes) + best trash", [("dinov2", "-", "full", "RBF SVM C=3")], [key(trash_res, 0)])
combo = pd.DataFrame(rows_)
print("best dishes:", key(dish_res, 0), f"F1 {dish_res.F1[0]:.3f}")
print("best trash: ", key(trash_res, 0), f"F1 {trash_res.F1[0]:.3f}")
combo.round(3)

best dishes: ('dinov2', '-', 'aug', 'logreg C=0.3') F1 0.878
best trash:  ('vith14', 'tiles', 'full', 'RBF SVM C=10') F1 0.910


,combination,CV macro F1,std
0,4-class baseline (dinov2+tiles RBF SVM),0.890,0.026
1,best dishes + best trash,0.894,0.025
2,top-5 dishes avg + top-5 trash avg,0.879,0.017
3,per chore: dinov2 RBF SVM C=3 (dishes) / dinov...,0.861,0.021
4,per chore: dinov2 logreg C=1 +rand10 (dishes) ...,0.870,0.025
5,best dishes + per-chore dinov2+tiles RBF SVM C...,0.885,0.020
6,per-chore dinov2 RBF SVM C=3 (dishes) + best t...,0.870,0.029


## 5. Honest estimate of 'pick the best per chore'
Nested check: inside each outer training fold, choose the dishes / trash model by an inner 3-fold CV over a shortlist (the top 15 per chore
from above), then score the chosen model on the outer fold. This tells us how much of the gain survives selection.

In [14]:
def nested(name, res, top=15, reps=3):
    comp, _, yb = chore_splits(name)
    ext = np.where((chore == name) & is_ext)[0]
    short = [key(res, i) for i in range(top)]
    cache = {k: (build(k[0], None if k[1] == "-" else k[1])[:NTR],
                 None if k[2] == "full" else views(k[0], None if k[1] == "-" else k[1], k[2])[:NTR]) for k in short}
    f1s, picks = [], []
    for r in range(reps):
        pred = np.zeros(NTR, int)
        for tr, va in StratifiedKFold(5, shuffle=True, random_state=100 + r).split(comp, yb[comp]):
            otr, ova = comp[tr], comp[va]
            inner = []
            for k in short:
                X, V = cache[k]; acc = []
                for itr, iva in StratifiedKFold(3, shuffle=True, random_state=r).split(otr, yb[otr]):
                    p = fit_predict(MODELS[k[3]], X, V, np.r_[otr[itr], ext], otr[iva], yb)
                    acc.append(f1_score(yb[otr[iva]], (p > 0.5).astype(int), average="macro"))
                inner.append(np.mean(acc))
            k = short[int(np.argmax(inner))]; picks.append(k)
            X, V = cache[k]
            pred[ova] = fit_predict(MODELS[k[3]], X, V, np.r_[otr, ext], ova, yb) > 0.5
        f1s.append(f1_score(yb[comp], pred[comp], average="macro"))
    return np.mean(f1s), np.std(f1s), pd.Series([" | ".join(p) for p in picks]).value_counts()

for name, res in (("dishes", dish_res), ("trash", trash_res)):
    m, s, picks = nested(name, res)
    print(f"{name}: top-of-grid F1 {res.F1[0]:.3f}  ->  nested (selection inside CV) F1 {m:.3f} ± {s:.3f}")
    print("  models picked in the outer folds:"); print(picks.head(5).to_string())

dishes: top-of-grid F1 0.878  ->  nested (selection inside CV) F1 0.899 ± 0.029
  models picked in the outer folds:
dinov2 | - | aug | logreg C=0.3           3
dinov2+siglip | - | full | extra trees    2
dinov2 | - | rand10 | logreg C=10         2
dinov2 | - | rand10 | linear SVM          2
dinov2+siglip | - | full | logreg C=10    2


trash: top-of-grid F1 0.910  ->  nested (selection inside CV) F1 0.849 ± 0.032
  models picked in the outer folds:
5-backbone concat | tiles | full | RBF SVM C=10    3
vith14 | tiles | full | RBF SVM C=10               2
eva02 | overflow | aug | logreg C=1                2
vith14 | tiles | full | linear SVM                 2
vitl14 | - | full | RBF SVM C=1                    1


## 6. Submission from the per-chore models
Router → chore; then the chosen dishes / trash model (trained on all photos of that chore incl. externals) decides Complete vs Incomplete.
Uses the **top-5 average per chore** (less winner's-curse than the single best). Compared with the 0.94155 public file and the current best-CV file.

In [15]:
def fit_full(name, keys):
    idx = np.where(chore == name)[0]; yb = y4 % 2; te = np.arange(NTR, len(ALL))
    ps = []
    for k in keys:
        extra = None if k[1] == "-" else k[1]
        X = build(k[0], extra); V = views(k[0], extra, k[2])
        Xtr, ytr = X[idx], yb[idx]
        if V is not None:
            Vt = V[idx]; keep = ~np.isnan(Vt).any(-1)
            Xtr = np.vstack([Xtr, Vt[keep]]); ytr = np.concatenate([ytr, np.repeat(ytr, Vt.shape[1])[keep.ravel()]])
        ps.append(p_incomplete(MODELS[k[3]]().fit(Xtr, ytr), X[te]))
    return np.mean(ps, 0)

pd_te = fit_full("dishes", [key(dish_res, i) for i in range(5)])
pt_te = fit_full("trash", [key(trash_res, i) for i in range(5)])
lab = np.where(test_chore == "dishes", np.where(pd_te > 0.5, CLASSES[1], CLASSES[0]), np.where(pt_te > 0.5, CLASSES[3], CLASSES[2]))
sub = pd.DataFrame({"id": test.id, "label": lab})
sub.to_csv(ROOT / "submission_per_chore.csv", index=False)
print(sub.label.value_counts().to_dict())
for f in ("submission_public0.94155_extratrees.csv", "submission_dinov2_tiles_svm.csv"):
    o = pd.read_csv(ROOT / f).set_index("id").label.reindex(sub.id).values
    diff = sub.id[o != sub.label].tolist()
    print(f"vs {f}: agree {np.mean(o == sub.label):.0%}; differs on test ids {diff}")
pd.concat([dish_res.assign(chore="dishes"), trash_res.assign(chore="trash")]).to_csv(ROOT / "logs/per_chore_cv.csv", index=False)

{'Washing Dishes|Incomplete': 12, 'Taking Out Trash|Incomplete': 9, 'Taking Out Trash|Complete': 8, 'Washing Dishes|Complete': 7}
vs submission_public0.94155_extratrees.csv: agree 86%; differs on test ids [4, 14, 21, 86, 95]
vs submission_dinov2_tiles_svm.csv: agree 94%; differs on test ids [4, 33]
